# Curso 1 — Datos y métricas### Detectar gatos para aprender a detectar fracturas---Este notebook no es parte del proyecto de radiografías. Es el **ensayo general**.Vamos a resolver un problema que tiene exactamente la misma forma que el nuestro,pero con datos que pesan poco, se descargan en dos minutos y entrenan en segundos.Cuando algo salga mal —y va a salir mal— vamos a poder probar veinte cosas en unaclase en vez de esperar media hora por cada intento.### El paralelo, que es lo importante| | Curso (CIFAR-10) | Proyecto real (FracAtlas) ||---|---|---|| Pregunta | ¿hay un gato en esta foto? | ¿hay una fractura en esta placa? || Imágenes | 60.000 de 32×32 px | 4.083 radiografías || Positivos | 10% son gatos | 17,6% tienen fractura || Modelo trivial acierta | **90%** | **82,4%** || Entrenar | segundos | minutos |Es el mismo problema: **detección binaria con clases desbalanceadas**.Todo lo que aprendas acá se usa igual allá.### Al terminar vas a saber1. Levantar un dataset y mirarlo antes de tocarlo2. Separar en entrenamiento, validación y test — y por qué son tres3. Por qué la *accuracy* miente y qué mirar en su lugar4. Elegir el umbral de decisión con un criterio, no por costumbre

---## 0. Preparar el entornoEn Colab: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU`.Para este notebook no hace falta GPU, pero conviene pedirla ahora porque elnotebook siguiente sí la usa.

In [ ]:
import sys, osfrom pathlib import Pathdef en_colab():    try:        import google.colab  # noqa        return True    except ImportError:        return Falseif en_colab():    CARPETA = "Deteccion-De-Fracturas-ETRR"    if not Path(CARPETA).exists():        !git clone -q https://github.com/resagri-fiuba/Deteccion-De-Fracturas-ETRR.git    RAIZ = Path("/content") / CARPETAelse:    RAIZ = Path.cwd()    while not (RAIZ / "src").exists() and RAIZ != RAIZ.parent:        RAIZ = RAIZ.parentsys.path.insert(0, str(RAIZ)); os.chdir(RAIZ)import numpy as np, pandas as pdimport matplotlib.pyplot as pltfrom src import cursonp.random.seed(42)print(f"Listo. Raíz del proyecto: {RAIZ}")

---## 1. Levantar los datos**CIFAR-10** son 60.000 fotos de 32×32 píxeles repartidas en 10 clases(avión, auto, pájaro, gato, ciervo, perro, rana, caballo, barco, camión),6.000 de cada una. Es público, anónimo, pesa unos 170 MB y es probablementeel conjunto de imágenes más usado de la historia para enseñar visión por computadora.La primera vez tarda un par de minutos en bajar; después queda en disco.

In [ ]:
X_ent, y_ent, X_test, y_test = curso.cargar_cifar10("datos")print(f"Entrenamiento : {X_ent.shape}   etiquetas {y_ent.shape}")print(f"Test          : {X_test.shape}   etiquetas {y_test.shape}")print()print(f"Una imagen    : {X_ent[0].shape}  →  alto, ancho, canales de color")print(f"Tipo de dato  : {X_ent.dtype}  →  enteros de 0 a 255")print(f"Clases        : {curso.CLASES_CIFAR}")

### Mirar antes de modelarLa misma regla que en el proyecto de radiografías. Nunca se entrena sobre datosque no se miraron.

In [ ]:
curso.grilla(X_ent[:32], filas=4, columnas=8,             titulo="32 imágenes de CIFAR-10, tal como vienen");

---## 2. De diez clases a una sola preguntaNuestro problema real no es "¿qué animal es?", es **"¿hay fractura, sí o no?"**.Así que convertimos CIFAR-10 en un problema del mismo tipo: **gato o no gato**.Una línea de código, y de golpe el dataset se desbalancea: los gatos pasan a serel 10% y todo lo demás el 90%. Igual que las fracturas.

In [ ]:
y_ent_bin  = curso.a_binario(y_ent,  clase=curso.GATO)y_test_bin = curso.a_binario(y_test, clase=curso.GATO)n = len(y_ent_bin)gatos = int(y_ent_bin.sum())print(f"Total          : {n:,}")print(f"Gatos          : {gatos:,}   ({gatos/n:.1%})")print(f"No gatos       : {n-gatos:,}   ({(n-gatos)/n:.1%})")print()print(f"⚠️  Un modelo que SIEMPRE dijera 'no es gato' acertaría el {(n-gatos)/n:.1%}.")print("    Ese es el número a vencer. Anotalo.")

In [ ]:
# Ahora la grilla con la etiqueta binaria: rojo = gatoi = np.random.permutation(len(X_ent))[:32]curso.grilla(X_ent[i], y_ent_bin[i], filas=4, columnas=8,             titulo="Gato vs. todo lo demás — así de desbalanceado está");

---## 3. Entrenamiento, validación y testEste es **el concepto que más se malentiende** y el que más resultados falsos produce.Se separan los datos en tres montones, y cada uno tiene una función distinta:| Montón | Para qué | Analogía ||---|---|---|| **Entrenamiento** (~70%) | El modelo aprende de acá | Los ejercicios que hacés todo el año || **Validación** (~15%) | Elegir entre modelos y ajustar | El simulacro antes del examen || **Test** (~15%) | Medir una única vez, al final | El examen de verdad |### ¿Por qué tres y no dos?Porque si probás treinta configuraciones y elegís la que mejor puntúa en un mismoconjunto, ese conjunto **deja de ser una medida independiente**: elegiste el modeloque le queda cómodo. Necesitás un tercero que no se haya usado nunca para decidir nada.> **La regla más importante del proyecto:** el conjunto de test se abre **una sola vez**,> cuando ya no se va a cambiar nada más. Cada vez que ajustás algo mirando el test,> el test se convierte en otro conjunto de validación y perdés tu única medida honesta.### Dos detalles técnicos que importan- **Estratificar** (`stratify=y`): que los tres montones tengan la misma proporción de  gatos. Sin esto, con clases desbalanceadas podés tener un montón con casi ningún positivo.- **Semilla fija** (`random_state=42`): que la partición sea siempre la misma.  Sin esto no se puede comparar nada con nada.

In [ ]:
from sklearn.model_selection import train_test_split# CIFAR-10 ya viene con su test separado de fábrica. Lo respetamos y no lo tocamos.# Del conjunto de entrenamiento sacamos la validación.X_tr, X_val, y_tr, y_val = train_test_split(    X_ent, y_ent_bin,    test_size=0.20,          # 20% para validación    stratify=y_ent_bin,      # misma proporción de gatos en los dos    random_state=42,         # siempre la misma partición)for nombre, y in [("entrenamiento", y_tr), ("validación", y_val), ("test", y_test_bin)]:    print(f"{nombre:15s} {len(y):>6,} imágenes   {y.mean():.1%} gatos")

Fijate que las tres proporciones son casi idénticas. Eso es lo que hizo `stratify`.Sin ese argumento podrían haber quedado en 8%, 12% y 10%, y las comparaciones entremontones dejarían de tener sentido.### El error que arruina proyectos: la fuga de informaciónEn CIFAR-10 cada foto es independiente, así que partir al azar está bien.**En el proyecto de radiografías no.** Si un mismo paciente tiene tres placas ydos caen en entrenamiento y una en test, el modelo ya vio la respuesta: las métricassuben y el modelo no mejoró nada.Por eso en la semana 2 del proyecto vamos a partir **por paciente**, no por imagen.Es la diferencia entre un resultado real y uno inflado.

---## 4. El baseline tontoAntes de entrenar nada, hay que saber contra qué comparamos. El punto de referenciaes el modelo más estúpido posible: el que siempre responde lo mismo.

In [ ]:
from sklearn.dummy import DummyClassifiertonto = DummyClassifier(strategy="most_frequent")tonto.fit(X_tr.reshape(len(X_tr), -1), y_tr)pred_tonta = tonto.predict(X_val.reshape(len(X_val), -1))from sklearn.metrics import accuracy_scoreprint(f"Accuracy del modelo que siempre dice 'no es gato': {accuracy_score(y_val, pred_tonta):.1%}")print()print("Este modelo no mira la imagen. No aprendió nada. Es un return.")print("Y si mañana alguien te muestra un modelo con 88% de accuracy acá,")print("en realidad te está mostrando algo PEOR que no hacer nada.")

---## 5. El primer modelo de verdad**Regresión logística.** Es el modelo más simple que aprende algo: le pasás los 3.072números de la imagen (32 × 32 × 3) y aprende un peso para cada uno, más un umbral.Es, literalmente, **una sola neurona**. En el notebook siguiente vamos a apilarmuchas de estas y eso va a ser una red neuronal.Dos preparaciones que necesita:- **Aplanar**: de una imagen (32, 32, 3) a una fila de 3.072 números. El modelo no  sabe que es una imagen; para él es una tabla con 3.072 columnas.- **Normalizar**: de enteros 0–255 a decimales 0–1. Los modelos entrenan mucho mejor  con números chicos y parejos.

In [ ]:
Xf_tr  = curso.aplanar_y_normalizar(X_tr)Xf_val = curso.aplanar_y_normalizar(X_val)print(f"Antes:   {X_tr.shape}   enteros 0–255")print(f"Después: {Xf_tr.shape}   decimales 0–1")

In [ ]:
from sklearn.linear_model import LogisticRegressionimport timet0 = time.time()modelo = LogisticRegression(max_iter=200)modelo.fit(Xf_tr, y_tr)print(f"Entrenado en {time.time()-t0:.0f} segundos")# predict devuelve 0/1 ; predict_proba devuelve la CONFIANZA, que es más útilpred_val  = modelo.predict(Xf_val)score_val = modelo.predict_proba(Xf_val)[:, 1]print(f"\nAccuracy: {accuracy_score(y_val, pred_val):.1%}")print(f"Baseline tonto: {(1 - y_val.mean()):.1%}")

### 🔴 Pará y mirá esos dos númerosEs muy probable que el modelo entrenado esté **apenas** por encima del tonto,o incluso por debajo. Y sin embargo "acierta el 90%".Esto no es un accidente del ejemplo: es lo que pasa siempre que hay desbalance.La accuracy está dominada por la clase mayoritaria y **no te dice nada** sobre loúnico que te importa, que es encontrar los positivos.Necesitamos otras métricas.

---## 6. Las métricas que sí sirvenTodo sale de cuatro números. Con dos respuestas posibles y dos realidades posibles,hay exactamente cuatro casos:

In [ ]:
curso.matriz_confusion(y_val, pred_val);

Con esos cuatro números se construye todo:| Métrica | Fórmula | La pregunta que responde ||---|---|---|| **Recall** (sensibilidad) | VP / (VP + FN) | De cada 100 gatos **reales**, ¿cuántos encuentra? || **Precisión** | VP / (VP + FP) | De cada 100 veces que **dice** gato, ¿cuántas acierta? || **F1** | media armónica | El equilibrio entre las dos || **Accuracy** | (VP+VN) / total | De cada 100 imágenes, ¿cuántas clasifica bien? ⚠️ engaña con desbalance |### Traducido a nuestro proyecto- **Recall bajo** = se le escapan fracturas. En medicina, es el error grave.- **Precisión baja** = manda a hacer estudios de más por falsas alarmas. Molesto, no grave.Los dos no se pueden maximizar juntos: subir uno baja el otro. **Elegir cuál priorizares una decisión clínica, no técnica**, y hay que poder defenderla en la presentación.

In [ ]:
m_logistica = curso.informe(y_val, pred_val, score_val, "regresión logística")

---## 7. El umbral: la perilla que casi nadie tocaEl modelo no devuelve "gato" o "no gato". Devuelve un **número entre 0 y 1**:su nivel de confianza. Recién después alguien decide a partir de qué valor seconsidera un sí.Ese valor por omisión es 0,5. **No tiene nada de especial.** Es una convención.Moverlo cambia por completo el comportamiento del sistema, sin reentrenar nada.

In [ ]:
print("Cómo cambia todo al mover el umbral:\n")print(f"{'umbral':>8} {'recall':>9} {'precisión':>11} {'accuracy':>10}")print("-" * 42)from sklearn.metrics import precision_score, recall_scorefor u in [0.05, 0.10, 0.20, 0.30, 0.50, 0.70, 0.90]:    p = (score_val >= u).astype(int)    print(f"{u:>8.2f} {recall_score(y_val,p,zero_division=0):>9.3f}"          f" {precision_score(y_val,p,zero_division=0):>11.3f}"          f" {accuracy_score(y_val,p):>10.3f}")

Bajar el umbral hace al modelo más desconfiado: encuentra más gatos (sube el recall)pero se equivoca más seguido cuando dice que sí (baja la precisión).**Las dos curvas que resumen todos los umbrales de una vez:**- La **curva ROC** y su AUC: qué tan bien separa el modelo las dos clases,  independientemente del umbral. Con desbalance fuerte tiende a verse  optimista.- La **curva precisión–recall**: la honesta cuando los positivos son pocos.  La línea punteada marca lo que sacaría el azar.

In [ ]:
curso.curvas(y_val, score_val);

### Elegir el umbral con un criterioEn un problema médico el razonamiento correcto va en este orden:1. Primero se decide **cuántos positivos se está dispuesto a dejar pasar**   (por ejemplo: queremos encontrar al menos el 90% de las fracturas).2. Después se mira **qué precisión queda** con ese recall.3. Y se decide si ese costo es aceptable.Nunca al revés.

In [ ]:
umbral, prec, rec = curso.umbral_para_recall(y_val, score_val, recall_minimo=0.90)pred_ajustada = (score_val >= umbral).astype(int)m_ajustada = curso.informe(y_val, pred_ajustada, score_val,                           f"logística con umbral {umbral:.2f}")

---## 8. El examen final: el test, una sola vezYa elegimos modelo y umbral usando **validación**. Recién ahora se abre el test.Este número es el que se reporta y no se toca más.

In [ ]:
Xf_test = curso.aplanar_y_normalizar(X_test)score_test = modelo.predict_proba(Xf_test)[:, 1]pred_test  = (score_test >= umbral).astype(int)m_test = curso.informe(y_test_bin, pred_test, score_test, "TEST (una sola vez)")

In [ ]:
# Todo junto, para comparartabla = pd.DataFrame([m_logistica, m_ajustada, m_test]).set_index("modelo")tabla.round(3)

---## 9. Qué nos llevamos a las radiografíasTodo. Literalmente todo lo de este notebook se usa igual en el proyecto real:1. **Mirar los datos antes de modelar.** Siempre.2. **Calcular el baseline tonto primero.** En FracAtlas es 82,4%. Sin ese número,   cualquier resultado parece bueno.3. **Tres montones, no dos**, y el test bajo llave hasta el final.   En radiografías, además, la partición va **por paciente**.4. **La accuracy no se reporta sola.** Recall, precisión y PR-AUC.5. **El umbral es una decisión, no un valor por omisión**, y se elige empezando   por cuántos positivos estamos dispuestos a perder.Lo único que cambia allá es que las imágenes pesan más, tardan más y son de personas.---## 🔧 Ejercicios1. **Fácil.** Cambiá la clase objetivo de `gato` a `perro` (`clase=5`).   ¿Cambian mucho las métricas? ¿Por qué te parece?2. **Fácil.** ¿Qué pasa si sacás `stratify=y_ent_bin` de la partición?   Corré varias veces con distintas semillas y mirá cómo se mueve el porcentaje de gatos.3. **Media.** Entrená solo con 1.000 imágenes en vez de 40.000. ¿Cuánto empeora?   Esto importa: en medicina casi nunca hay 40.000 casos etiquetados.4. **Media.** Buscá las 10 imágenes que el modelo clasificó como gato con más   confianza y que **no** eran gatos. Miralas. ¿Se parecen entre sí?   (Pista: `np.argsort(score_val)` y filtrar por `y_val == 0`.)5. **Difícil.** Reemplazá la regresión logística por `RandomForestClassifier`.   ¿Mejora? ¿Cuánto más tarda? ¿Vale la pena el cambio?---**Siguiente:** `curso/02_redes_neuronales.ipynb` — por qué una red convolucionalgana por goleada en imágenes, y cómo se entrena una.